In [25]:
import os
# import numpy as np
# import torch
import pandas as pd
from numpy import kron
import functools as fu
from itertools import *
from itertools import product
from numpy.linalg import cholesky, eig
from qiskit.quantum_info import random_density_matrix
from qutip import Qobj, fidelity
from scipy.stats import norm
from qiskit.quantum_info import random_statevector
from array_api_compat import array_namespace
from data_gen import *
import platform


In [31]:
def get_compute_backend(use_gpu=True):
    """
    Dynamically detects available hardware and returns the Array API namespace (xp) 
    and the target device. This prevents ModuleNotFoundErrors across environments.
    """
    import array_api_compat.numpy as xp_np
    
    if not use_gpu:
        print("Backend: NumPy (CPU - Forced)")
        return xp_np, "cpu"
    
    # Check for Apple Silicon (Macs)
    if platform.system() == "Darwin" and platform.machine() == "arm64":
        try:
            import torch
            import array_api_compat.torch as xp_torch
            print("Backend: PyTorch (Apple MPS)")
            return xp_torch, torch.device("mps")
        except ImportError:
            pass
            
    # Check for NVIDIA GPUs (Cluster)
    try:
        import cupy
        import array_api_compat.cupy as xp_cupy
        print("Backend: CuPy (NVIDIA CUDA)")
        return xp_cupy, cupy.cuda.Device(0)
    except ImportError:
        pass
        
    print("Backend: NumPy (CPU - Fallback)")
    return xp_np, "cpu"

In [32]:
get_compute_backend(True)

Backend: PyTorch (Apple MPS)


(<module 'array_api_compat.torch' from '/opt/miniconda3/envs/nbqss26/lib/python3.12/site-packages/array_api_compat/torch/__init__.py'>,
 device(type='mps'))

In [2]:
!pip list | grep qiskit

qiskit                  2.3.1
qiskit-aer              0.17.2
qiskit-algorithms       0.4.0
qiskit-ibm-runtime      0.49.0
qiskit-qasm3-import     0.6.0


## 1. Depolarization channel

We provide a local and a global depolarizing channel to introduce an extra physical channel. These files can be used in step 3 during the data generation and linear inversion pre-processing.

The channel that we use for noise in this case is $$\rho_{final} = (1-p)\rho + p/3 (X\rho X^\dagger + Y \rho Y^\dagger + Z \rho Z^\dagger)$$ 

## 2. Support functions:
1. the vectorization of the Cholesky matrices $vec(C_{ij})$.
2. positive semidefinite brute-force approximation of non physical matrices $\rho_{LI}$(eigenvaluescheck, PureEigenvaluesCheck)


## 3. Random states generation, pre-processing, training dataset preparation.

Algorithm:

1. generate random matrices
2. calculate Born values
3. approximate with fine statistics, fixing the parameter "trials" (for SICS), or normalizing the Gaussian noise (for Pauli)
4. using linear inversion to reconstruct the density matrix, and approximate it to the close positive definite approximation thereof, killing the negative eigenvalues
5. vectorize the original matrix and the new, experimental one. This is our training dataset

## State Generation

### Random Pure

In [18]:
random_pure_data = generate_data(4, 2, 1000, "random_pure")

random_pure_data['data_array']

Mean fidelity: 83.4409%


array([[ 0.31739026,  0.20437777,  0.18890502, ..., -0.02228004,
        -0.01219313,  0.0142565 ],
       [ 0.25536463,  0.1633818 ,  0.2041786 , ...,  0.04086512,
         0.02940488,  0.010251  ],
       [ 0.19338697,  0.25261319,  0.21199271, ..., -0.0136073 ,
        -0.02173172,  0.00146358],
       ...,
       [ 0.2189329 ,  0.21215379,  0.12289829, ..., -0.02019118,
        -0.00667955, -0.06955426],
       [ 0.30115977,  0.23237573,  0.1184623 , ...,  0.00999634,
        -0.00750549, -0.05350352],
       [ 0.30953479,  0.1963446 ,  0.18566765, ..., -0.02852836,
         0.0332908 ,  0.01076256]], shape=(1000, 512))

### Random Mixed

In [19]:
random_mixed_data = generate_data(4, 2, 1000, "random_mixed")

random_mixed_data['data_array']

Mean fidelity: 81.7407%


array([[ 2.70540893e-01,  2.08630711e-01,  1.64099112e-01, ...,
         2.45639555e-16,  7.62329653e-17, -7.94093388e-17],
       [ 2.82894671e-01,  2.45352730e-01,  1.95054173e-01, ...,
        -1.91641204e-16,  3.26901778e-16, -5.29395592e-17],
       [ 2.98700482e-01,  1.91326186e-01,  2.86397368e-01, ...,
        -3.30342849e-16, -9.31736242e-17,  1.92699996e-16],
       ...,
       [ 2.66113162e-01,  2.41712704e-01,  2.14215085e-01, ...,
         2.96461532e-17,  2.75285708e-17,  3.38813179e-17],
       [ 2.05480576e-01,  2.48790607e-01,  2.13003948e-01, ...,
        -4.23516474e-18,  2.47757137e-16,  8.47032947e-17],
       [ 2.15979815e-01,  2.29631394e-01,  1.55630857e-01, ...,
         3.36695597e-16, -1.86347248e-16, -2.31610572e-17]],
      shape=(1000, 512))

### Haar-Random Pure

In [20]:
haar_random_data = generate_data(4, 2, 1000, "haar_random")

haar_random_data['data_array']

Mean fidelity: 123.7593%


array([[ 0.22355288,  0.12352397,  0.11943021, ..., -0.00602107,
         0.00436106,  0.00195621],
       [ 0.20742565,  0.26003128,  0.15897484, ..., -0.00861448,
        -0.01622076,  0.03909462],
       [ 0.24789257,  0.15348026,  0.11408944, ...,  0.01340708,
         0.01359395,  0.00289395],
       ...,
       [ 0.30938518,  0.13872331,  0.17297606, ...,  0.00191701,
        -0.00874468,  0.00759019],
       [ 0.4121342 ,  0.16832261,  0.1225186 , ...,  0.0098841 ,
        -0.0020694 ,  0.00176869],
       [ 0.22716287,  0.17065807,  0.19216251, ...,  0.00120839,
        -0.00557206, -0.00045031]], shape=(999, 512))

### Random Product State Generation

In [23]:
random_product_data = generate_data(4, 2, 1000, "random_product")

random_product_data['data_array']

Mean fidelity: 83.4131%


array([[ 0.20957977,  0.23812889,  0.16510986, ..., -0.02542164,
        -0.00786233,  0.06375738],
       [ 0.13711751,  0.18612263,  0.18967836, ...,  0.01925325,
         0.10245747, -0.0160449 ],
       [ 0.229967  ,  0.24375927,  0.16652125, ...,  0.03894673,
         0.06582316, -0.01038235],
       ...,
       [ 0.23392962,  0.20982425,  0.15854502, ..., -0.01854661,
        -0.00149426, -0.0228446 ],
       [ 0.13834345,  0.53321892,  0.1876844 , ...,  0.00590502,
        -0.03415718, -0.00209161],
       [ 0.18444844,  0.16568153,  0.19527377, ..., -0.0039713 ,
        -0.00783036,  0.0074934 ]], shape=(1000, 512))